In [3]:
from net2brain.feature_extraction import FeatureExtractor
import torch
import open_clip

import cv2
import os, glob
import numpy as np
import joblib
from pathlib import Path

In [4]:
def video_for(stim_id):
    positions = [vid for vid in vids if f'/{stim_id}.mp4' in vid]
    return positions[0]

def dump_frames_for_batch(stim_ids, out_dir):
    for f in glob.glob(os.path.join(out_dir, '*.jpg')):
        os.remove(f)
    os.makedirs(out_dir, exist_ok=True)
    for sid in stim_ids:
        cap = cv2.VideoCapture(video_for(sid))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        picks = np.linspace(0, max(total - 1, 0), N_FRAMES).astype(int)
        for k, fno in enumerate(picks):
            path = os.path.join(out_dir, f'{int(sid):05d}_f{k:02d}.jpg')
            if os.path.exists(path):
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fno))
            ok, frame = cap.read()
            if ok:
                cv2.imwrite(path, frame)
        cap.release()

def consolidate_to_video_level(raw_npz_path, output_path):
        z = np.load(raw_npz_path)
        video_features = {}
        
        all_sids = set(k.split('_')[0] for k in z.files)
        
        for sid in all_sids:
            keys = [k for k in z.files if k.startswith(f'{sid}_')]
            frame_feats = [z[k].flatten() for k in keys]
            video_features[sid] = np.mean(frame_feats, axis=0)
            
        # Save as a clean video-level archive
        np.savez_compressed(output_path, **video_features)
        print(f"Saved video-level features for {len(video_features)} videos to {output_path}")

# Process videos and extract Alex, Dino, and OpenCLIP features
## Store them under eeg-mind-reading-challenge/artifacts

## 1. PREPARE TRAINING SET

In [ ]:
BASE_DIR     = Path('../eeg-mind-reading-challenge')
STIM_DIR     = BASE_DIR / 'stimuli_train'  
FEAT_DIR     = BASE_DIR / 'features'       
FRAME_DIR    = BASE_DIR / 'frames'   
TEMP_FEAT    = BASE_DIR / 'temp_features' 
TRAIN_DIR    = BASE_DIR / 'trn'

N_FRAMES   = 8                                             
BATCH_SIZE = 200                                         

os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(TEMP_FEAT, exist_ok=True)
os.makedirs(FRAME_DIR, exist_ok=True)

vids = sorted(list(STIM_DIR.glob('*.mp4')))
print(f"Found {len(vids)} total videos in {STIM_DIR}")
all_stim_ids = np.arange(1, len(vids) + 1)

fxs = []
LAYERS = []

fx = FeatureExtractor(model='AlexNet', netset='Standard', device='cpu')   # 'cuda' if available
LAYER = 'features.3'
fxs.append(fx)
LAYERS.append(LAYER)

dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').eval()
fx = FeatureExtractor(model=dino_model, device='cpu')   # 'cuda' if available
LAYER = 'blocks.6'
fxs.append(fx)
LAYERS.append(LAYER)

clip_model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
clip_model.eval()
fx = FeatureExtractor(model=clip_model, device='cpu')
LAYER = 'visual.transformer.resblocks.7'
fxs.append(fx)
LAYERS.append(LAYER)

for fx, LAYER in zip(fxs, LAYERS):
    master_features = {}

    for i in range(0, len(all_stim_ids), BATCH_SIZE):
        batch_ids = all_stim_ids[i:i + BATCH_SIZE]
        print(f"\nProcessing batch: videos {i+1} to {min(i + BATCH_SIZE, len(all_stim_ids))}...")
        
        dump_frames_for_batch(batch_ids, FRAME_DIR)
        
        fx.extract(data_path=FRAME_DIR, layers_to_extract=[LAYER], save_path=TEMP_FEAT)
        
        batch_npz_path = TEMP_FEAT / f'consolidated_{LAYER}.npz'
        if os.path.exists(batch_npz_path):
            z = np.load(batch_npz_path)
            for k in z.files:
                master_features[k] = z[k]
            os.remove(batch_npz_path)  

    # 4. Save the final consolidated .npz file
    final_npz_path = FEAT_DIR / f'consolidated_{LAYER}.npz'
    np.savez_compressed(final_npz_path, **master_features)
    print(f"\nSuccessfully extracted and saved all features to {final_npz_path} without memory crashes!")

    consolidate_to_video_level(f'./eeg-mind-reading-challenge/features/consolidated_{LAYER}.npz', f'./eeg-mind-reading-challenge/features/video_{LAYER}.npz')

## 1. PREPARE TEST SET

In [ ]:
BASE_DIR         = Path('../eeg-mind-reading-challenge')
TEST_DIR         = BASE_DIR / 'tst'
STIM_TEST_DIR    = BASE_DIR / 'stimuli_test'
FRAME_TEST_DIR   = BASE_DIR / 'frames_test'
FEAT_TEST_DIR    = BASE_DIR / 'features_test'
TEMP_FEAT        = BASE_DIR / 'temp_features'
TEST_STIM        = BASE_DIR / 'tst_stim'

N_FRAMES         = 8                                             
BATCH_SIZE       = 200

os.makedirs(FEAT_TEST_DIR, exist_ok=True)
os.makedirs(TEMP_FEAT, exist_ok=True)
os.makedirs(FRAME_TEST_DIR, exist_ok=True)

vids = sorted(list(STIM_TEST_DIR.glob('*.mp4')))
print(len(vids), 'videos found')
all_stim_ids = np.arange(1001, 1001 + len(vids))

fxs = []
LAYERS = []

fx = FeatureExtractor(model='AlexNet', netset='Standard', device='cpu')   # 'cuda' if available
LAYER = 'features.3'
fxs.append(fx)
LAYERS.append(LAYER)

dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').eval()
fx = FeatureExtractor(model=dino_model, device='cpu')   # 'cuda' if available
LAYER = 'blocks.6'
fxs.append(fx)
LAYERS.append(LAYER)

clip_model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
clip_model.eval()
fx = FeatureExtractor(model=clip_model, device='cpu')
LAYER = 'visual.transformer.resblocks.7'
fxs.append(fx)
LAYERS.append(LAYER)

for fx, LAYER in zip(fxs, LAYERS):
    master_features = {}

    for i in range(0, len(all_stim_ids), BATCH_SIZE):
        batch_ids = all_stim_ids[i:i + BATCH_SIZE]
        print(f"\nProcessing batch: videos {i+1} to {min(i + BATCH_SIZE, len(all_stim_ids))}...")
        
        dump_frames_for_batch(batch_ids, FRAME_TEST_DIR)

        fx.extract(data_path=FRAME_TEST_DIR, layers_to_extract=[LAYER], save_path=TEMP_FEAT)
        
        batch_npz_path = TEMP_FEAT / f'consolidated_{LAYER}.npz'
        if os.path.exists(batch_npz_path):
            z = np.load(batch_npz_path)
            for k in z.files:
                master_features[k] = z[k]
            os.remove(batch_npz_path) 

    final_npz_path = FEAT_TEST_DIR / f'consolidated_{LAYER}.npz'
    np.savez_compressed(final_npz_path, **master_features)
    print(f"\nSuccessfully extracted and saved all features to {final_npz_path} without memory crashes!")

    consolidate_to_video_level(f'./eeg-mind-reading-challenge/features_test/consolidated_{LAYER}.npz', f'./eeg-mind-reading-challenge/features_test/video_{LAYER}.npz')

102 videos found


Using cache found in /Users/yetkin/.cache/torch/hub/facebookresearch_dinov2_main
/Users/yetkin/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/yetkin/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/yetkin/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
/Users/yetkin/Desktop/Hackathon/.venv/lib/python3.11/site-packages/net2brain/feature_extraction.py:104: UserWarning: If you add your own model you can also select our own: 
Preprocessing Function (preprocessor) 
Extraction Function (extraction_function) 
Feature Cleaner (feature_cleaner) 
  warnings.warn("If you add your own model you ca


Processing batch: videos 1 to 102...


Processing files: 100%|██████████| 816/816 [00:18<00:00, 44.34it/s, subfiles=1/1]


Consolidating data per layer...


...


Successfully extracted and saved all features to ./eeg-mind-reading-challenge/features_test/consolidated_blocks.6.npz without memory crashes!
Saved video-level features for 102 videos to ./eeg-mind-reading-challenge/features_test/video_blocks.6.npz

Processing batch: videos 1 to 102...


Processing files: 100%|██████████| 816/816 [00:38<00:00, 20.99it/s, subfiles=1/1]


Consolidating data per layer...


100%|██████████| 816/816 [00:00<00:00, 1711.67it/s]



Successfully extracted and saved all features to ./eeg-mind-reading-challenge/features_test/consolidated_visual.transformer.resblocks.7.npz without memory crashes!
Saved video-level features for 102 videos to ./eeg-mind-reading-challenge/features_test/video_visual.transformer.resblocks.7.npz
